In [9]:
import pandas as pd

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [10]:
import numpy as np
import re
import torch
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from scipy.special import softmax

In [11]:
conflicts = train_df[train_df.duplicated(['text'], keep=False)].groupby('text')['target'].nunique()
train_df = train_df[~train_df['text'].isin(conflicts[conflicts > 1].index)].drop_duplicates(subset=['text']).copy()

def minimal_preprocess(text):
    text = re.sub(r'http\S+', 'HTTPURL', text)
    text = re.sub(r'@\w+', '@USER', text)
    return re.sub(r'\s+', ' ', text).strip()

train_df['text'] = train_df['text'].apply(minimal_preprocess)
test_df['text'] = test_df['text'].apply(minimal_preprocess)

In [12]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}

In [13]:
MODEL_NAME = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, normalization=True)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_tokenized = Dataset.from_pandas(test_df[['id', 'text']]).map(tokenize_function, batched=True)

# 1. GENEROWANIE PSEUDO-ETYKIET
print("Faza 1: Trening modelu bazowego i ocena zbioru testowego...")
full_train_tokenized = Dataset.from_pandas(train_df[['text', 'target']].rename(columns={'target': 'label'})).map(tokenize_function, batched=True)

model_base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
args_base = TrainingArguments(output_dir="./tmp", num_train_epochs=2, per_device_train_batch_size=16, learning_rate=2e-5, save_strategy="no", report_to="none")

trainer_base = Trainer(model=model_base, args=args_base, train_dataset=full_train_tokenized, processing_class=tokenizer, data_collator=data_collator)
trainer_base.train()

test_probs = softmax(trainer_base.predict(test_tokenized).predictions, axis=1)

# 2. FILTROWANIE "PEWNIAKÓW"
confidence_threshold = 0.95
max_probs = np.max(test_probs, axis=1)
confident_idx = np.where(max_probs >= confidence_threshold)[0]

pseudo_df = test_df.iloc[confident_idx].copy()
pseudo_df['target'] = np.argmax(test_probs[confident_idx], axis=1)

# 3. ROZBUDOWA ZBIORU
augmented_train_df = pd.concat([train_df[['text', 'target']], pseudo_df[['text', 'target']]], ignore_index=True)
print(f"\nDodano {len(pseudo_df)} pewnych tweetów z testu. Nowy rozmiar treningowy: {len(augmented_train_df)}")

# 4. DOCELOWY TRENING K-FOLD NA POWIĘKSZONYCH DANYCH
print("\nFaza 2: Walidacja krzyżowa (K-Fold) na wzmocnionym zbiorze...")
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
final_test_probs = np.zeros((len(test_df), 2))

args_kfold = TrainingArguments(output_dir="./results_pseudo", num_train_epochs=3, per_device_train_batch_size=16, weight_decay=0.01, warmup_steps=50, learning_rate=2e-5, save_strategy="no", report_to="none")

for fold, (train_idx, val_idx) in enumerate(skf.split(augmented_train_df, augmented_train_df['target'])):
    print(f"\nTrenowanie Foldu {fold + 1}/3")
    train_fold = augmented_train_df.iloc[train_idx].rename(columns={'target': 'label'})
    val_fold = augmented_train_df.iloc[val_idx].rename(columns={'target': 'label'})

    tokenized_train = Dataset.from_pandas(train_fold).map(tokenize_function, batched=True)
    tokenized_val = Dataset.from_pandas(val_fold).map(tokenize_function, batched=True)

    model_fold = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    trainer_fold = Trainer(model=model_fold, args=args_kfold, train_dataset=tokenized_train, eval_dataset=tokenized_val, processing_class=tokenizer, data_collator=data_collator)

    trainer_fold.train()
    final_test_probs += softmax(trainer_fold.predict(test_tokenized).predictions, axis=1) / skf.n_splits


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/843k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.91M [00:00<?, ?B/s]

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Map:   0%|          | 0/3263 [00:00<?, ? examples/s]

... (23 outputs left out)

Map:   0%|          | 0/2899 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.404515
1000,0.273624


In [14]:
final_preds = np.argmax(final_test_probs, axis=1)
pd.DataFrame({'id': test_df['id'], 'target': final_preds}).to_csv('submission_pseudo_labeling.csv', index=False)